+ this notebook generates Backend-required files to view a project with the DataDiVR (preview or VR)
+ STEP 1 and the "create a graph" section contains a template graph writing a required format (json) to then use the generate-project functions of the DataDiVR backend

+ STEP 2 to actually generate BACKEND project files.

In [1]:
import networkx as nx
import json 
import os

# these are the two functions one needs to create a JSON file to upload and create the project in the backend 
import nx2json as nx2j
import uploaderGraph as uG


## How this is meant to be used:
+ Create an nx.Graph Object 

+ set attributes in the nx.Graph (optional, all can be empty) e.g. node positions ("pos") and colors ("nodecolor") and link colors ("linkcolor")

+ use the "create_project" function (down below to generate your project for the platform)

# make VR project 

In [3]:
# make nx.graph from tsv file 
import pandas as pd
df_nodes = pd.read_csv("temp-files/students/positions_per_location.tsv", sep="\t")  

# add color columns r g b a (ideally this is in the file already) 
df_nodes['r'] = 100
df_nodes['g'] = 150
df_nodes['b'] = 200
df_nodes['a'] = 200

# get links from DATA (chloe) source 
df_links_uniprot = pd.read_csv("temp-files/students/consensus_ppi_bioplex_biogrid_intact_huri_edgelist.tsv", sep="\t", header=None)

df_nodes


,Node ID,x,y,z,r,g,b,a
0,Q6UWY0,-222.357100,1687.018840,3782.702621,100,150,200,200
1,Q9H0U3,3961.390715,2148.441916,-2145.565466,100,150,200,200
2,Q05707,-2002.859816,2023.204403,4129.907382,100,150,200,200
3,P20036,145.879351,1079.085284,4900.218594,100,150,200,200
4,Q13586,3634.210858,-1635.926845,-385.922313,100,150,200,200
...,...,...,...,...,...,...,...,...
3467,Q8N9T8,828.351138,1259.441155,566.447491,100,150,200,200
3468,P42677,1307.502933,1059.026024,548.535570,100,150,200,200
3469,Q9BQ39,452.970782,1189.869517,291.792645,100,150,200,200
3470,Q8N5L8,539.443722,397.857802,-68.834785,100,150,200,200


In [5]:
G = nx.Graph()
G.add_nodes_from(df_nodes['Node ID'])
for i, row in df_nodes.iterrows():
    G.nodes[row['Node ID']]['pos'] = [row['x'], row['y'], row['z']]
    G.nodes[row['Node ID']]['nodecolor'] = [row['r'], row['g'], row['b'], row['a']]
G.add_edges_from(df_links_uniprot.values)

# add positions to Graph pos attributes 
# go through all graph nodes - and if none 
nx.set_node_attributes(G, {row['Node ID']: [row['x'], row['y'], row['z']] for i, row in df_nodes.iterrows()}, 'pos')

print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Graph has 19555 nodes and 313667 edges.


In [62]:
df_n_comp = pd.read_csv("temp-files/students/protein_location_HPA_GO.tsv", sep="\t")
df_n_comp

,protein,location,source(s)
0,DIAPH2,Cytosol,GO
1,ACTL8,Cytosol,HPA
2,FZD2,Primary cilium,HPA
3,PKN2,Plasma membrane,HPA;GO
4,CREB3,Endoplasmic reticulum,GO
...,...,...,...
28013,IGKV2D-28,Plasma membrane,GO
28014,CLDN1,Plasma membrane,GO
28015,ANXA6,Primary cilium,HPA
28016,SHC4,Cytosol,HPA


In [63]:
# from UNITPROT to gene symbol 

# modify ids from uniprot to gene symbol 
df_uniprot_genesym = pd.read_csv("temp-files/students/d_uniprot_to_symbol_20251120.tsv", sep="\t")
df_uniprot_genesym_dict = pd.Series(df_uniprot_genesym.symbol.values,index=df_uniprot_genesym.uniprotid).to_dict()

# replace node ids in graph
mapping_dict = {}
for n in G.nodes():
    if n in df_uniprot_genesym_dict:
        mapping_dict[n] = df_uniprot_genesym_dict[n]
G = nx.relabel_nodes(G, mapping_dict)

print(f"After relabeling, graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

After relabeling, graph has 18346 nodes and 308377 edges.


In [64]:
# node attributes
node_attributes = {}

for n in G.nodes(): 
    comp = df_n_comp[df_n_comp['protein'] == n]['location'].values
    if len(comp) > 0:
        node_attributes[n] = {'location': comp[0]}
    else:
        node_attributes[n] = {'location': 'unknown'}


In [65]:
node_attributes

{'STIM1': {'location': 'Plasma membrane'},
 'AGPAT4': {'location': 'Endoplasmic reticulum'},
 'P32969': {'location': 'unknown'},
 'RPL27A': {'location': 'Nucleoplasm'},
 'NCK1': {'location': 'Plasma membrane'},
 'APOA2': {'location': 'Cytosol'},
 'BSG': {'location': 'Golgi apparatus'},
 'EMC10': {'location': 'Endoplasmic reticulum'},
 'PTPN1': {'location': 'Plasma membrane'},
 'TMED9': {'location': 'Endoplasmic reticulum'},
 'GPR37': {'location': 'Plasma membrane'},
 'DNAJA1': {'location': 'Cytosol'},
 'ORMDL1': {'location': 'Endoplasmic reticulum'},
 'TMPRSS3': {'location': 'Endoplasmic reticulum'},
 'PTPN5': {'location': 'Nucleoplasm'},
 'TPTE': {'location': 'Cytosol'},
 'BET1': {'location': 'Golgi apparatus'},
 'DDOST': {'location': 'Plasma membrane'},
 'RNF19B': {'location': 'Endoplasmic reticulum'},
 'ATXN3': {'location': 'Nucleoplasm'},
 'ERLIN1': {'location': 'Endoplasmic reticulum'},
 'RNF185': {'location': 'Endoplasmic reticulum'},
 'COL5A1': {'location': 'Endoplasmic reticulu

In [66]:
# get unique compartments
unique_compartments = df_n_comp['location'].unique().tolist()
# add unknown
unique_compartments.append('unknown')
print(unique_compartments)

# make color map for compartments based on "viridis"colormap
import matplotlib.pyplot as plt
cmap = plt.get_cmap('viridis', len(unique_compartments))
compartment_color_map = {}
for i, comp in enumerate(unique_compartments):
    color = cmap(i)  # RGBA tuple
    # Convert to 0-255 range for r, g, b, a
    compartment_color_map[comp] = [int(color[0]*255), int(color[1]*255), int(color[2]*255), 200]
print(compartment_color_map)


['Cytosol', 'Primary cilium', 'Plasma membrane', 'Endoplasmic reticulum', 'Nuclear membrane', 'Centrosome', 'Golgi apparatus', 'Nucleoplasm', 'Nucleoli', 'Mitochondria', 'Intermediate filaments', 'Actin filaments', 'Microtubules', 'unknown']
{'Cytosol': [68, 1, 84, 200], 'Primary cilium': [72, 28, 110, 200], 'Plasma membrane': [69, 53, 128, 200], 'Endoplasmic reticulum': [61, 76, 137, 200], 'Nuclear membrane': [51, 96, 141, 200], 'Centrosome': [43, 116, 142, 200], 'Golgi apparatus': [35, 135, 141, 200], 'Nucleoplasm': [30, 153, 138, 200], 'Nucleoli': [37, 171, 129, 200], 'Mitochondria': [64, 189, 114, 200], 'Intermediate filaments': [103, 204, 92, 200], 'Actin filaments': [151, 216, 62, 200], 'Microtubules': [205, 224, 29, 200], 'unknown': [253, 231, 36, 200]}


In [73]:
# add node attributes to graph as annotation
for n in G.nodes():
    if n in node_attributes:
        G.nodes[n]['annotation'] = dict(location=node_attributes[n]['location'])
        G.nodes[n]['nodecolor'] = compartment_color_map[G.nodes[n]['annotation']['location']]

In [74]:
# print Graph node attributes
for n in G.nodes(data=True):
    print(n)

('STIM1', {'pos': [-3287.009265041689, -5510.379405545829, -5171.142239086409], 'nodecolor': [69, 53, 128, 200], 'annotation': {'location': 'Plasma membrane'}})
('AGPAT4', {'pos': [-291.21538147795275, 4978.437695495098, -492.1721103398928], 'nodecolor': [61, 76, 137, 200], 'annotation': {'location': 'Endoplasmic reticulum'}})
('P32969', {'pos': [1064.3113626289314, 1142.134944984536, 59.42814693103409], 'nodecolor': [253, 231, 36, 200], 'annotation': {'location': 'unknown'}})
('RPL27A', {'pos': [2141.6079254793417, 3283.753560757641, -1058.2937816410606], 'nodecolor': [30, 153, 138, 200], 'annotation': {'location': 'Nucleoplasm'}})
('NCK1', {'annotation': {'location': 'Plasma membrane'}, 'nodecolor': [69, 53, 128, 200], 'pos': [0.0, 0.0, 0.0]})
('APOA2', {'pos': [-3112.915356923353, 1815.125125908884, 3464.670931179256], 'nodecolor': [68, 1, 84, 200], 'annotation': {'location': 'Cytosol'}})
('BSG', {'pos': [-5425.812800384063, 2262.549986508764, -91.22588680456954], 'nodecolor': [35, 

In [75]:
# for nodes without position assign position 0,0,0  

nodes_without_pos = [n for n in G.nodes() if 'pos' not in G.nodes[n]]
print(f"Nodes without position: len(nodes_without_pos) = {len(nodes_without_pos)}")
for n in G.nodes():
    if 'pos' not in G.nodes[n]:
        G.nodes[n]['pos'] = [0.0, 0.0, 0.0]


Nodes without position: len(nodes_without_pos) = 0


In [83]:
# link colors 
for u, v in G.edges():
    G.edges[u, v]['linkcolor'] = [60,60,60, 100]

✅ Connected to /main


# make VR project with Graph 

In [84]:
print("Number of nodes: ", len(G.nodes()))
print("Number of Links: ", len(G.edges()))

# ===============================================
# GRAPH NAME AND DESCRIPTION - a string each
# ===============================================

G.graph['projectname'] = "0_SpatialPPI_Elias"
G.graph['info'] = "A spatial PPI graph for testing purposes. Number of nodes: "+str(len(G.nodes()))+", Links: "+ str(len(G.edges()))+"."

Number of nodes:  18346
Number of Links:  308377


In [85]:
# set node names (optional)
#for n in G.nodes():
#    G.nodes[n]['name'] = 

In [86]:
nx2j.create_project(G)

Successfully created the directory static/projects/0_SpatialPPI_Elias 
PROGRESS: loaded graph JSON...
PROGRESS: stored graph data...
PROGRESS: stored layouts...
PROGRESS: stored node info...
PROGRESS: made node position textures...
PROGRESS: made textures for node colors...
PROGRESS: made textures for links...
PROGRESS: made textures for linkcolors...
PROGRESS: writing json files for project and nodes...
Project created successfully.


✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main


# REALTIME 

In [80]:
import networkx as nx 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

In [81]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()

#client.disconnect()

✅ Connected to /main


✅ Connected to /main
✅ Connected to /main


In [79]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, '0_SpatialPPI_Elias'),
 (1, 'AE_Memes_2022'),
 (2, 'ARS23_memes'),
 (3, 'ByzNet-1400-people-only'),
 (4, 'ByzNet_PxL'),
 (5, 'CDK5'),
 (6, 'CircLadderGraph-xsmall'),
 (7, 'diffusion'),
 (8, 'JSON_autocore'),
 (9, 'JSON_barbellgraph'),
 (10, 'JSON_Zachary'),
 (11, 'Microplastics_HumanHealth'),
 (12, 'Pesticides_HumanHealth'),
 (13, 'PG_NEW'),
 (14, 'Powergrid_Europe'),
 (15, 'PPI_brain_infarction'),
 (16, 'PPI_joel_daniel_aryan'),
 (17, 'PPI_joel_daniel_aryan_C'),
 (18, 'PPI_joel_daniel_aryan_test_1'),
 (19, 'PPI_networkcartoGRAPHs'),
 (20, 'Realtime-project'),
 (21, 'Sphere_Torus'),
 (22, 'Teapot'),
 (23, 'Template'),
 (24, 'test'),
 (25, 'TheMandelbulb_edges'),
 (26, 'XX')]

In [82]:
# select a project to work with
sel_id = 0
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()
session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)

Session Graph loaded from project folder. 
Project name:  0_SpatialPPI_Elias
Data: Nodes: 18346 Links: 308377


# THE FOLLOWING SECTION IS JUST FOR BACKGROUND INFO - no need to run
This is how the graph is storing all information given nx.Graph(s).

In [ ]:
'''

{
----------------------------------------
THIS IS THE GENERAL GRAPH INFO SECTION
----------------------------------------
  "directed": false,
  "multigraph": false,
  "projectname": "Testgraph",
  "info": "A toy graph for testing purposes. Number of nodes: 10, Links: 43.",
  "graphlayouts": [
      "layout1-spring",
      "layout2-spring",
      "layout3-spring",
      "layout4-clusters"
  ],
  "annotationTypes": true,
  "nodes": [
   ----------------------------------------
   contains all nodes of the project
   ----------------------------------------
      {
          "id": 0,
          "name": nodename-x,
          "annotation": 
                {
                    "annot1": [
                        "lambda",
                        "alpha",
                        "zeta",
                        "theta"
                    ],
                    "annot2": [
                        "delta",
                        "nu"
                    ],
                    "annot3": [
                        "mu",
                        "gamma"
                    ]
          }
      },....
  ],
  "links": [
   ----------------------------------------
   contains all links of the project
   ----------------------------------------
      {
          "id": 0,
          "source": 0,
          "target": 1
      },
      {
          "id": 1,
          "source": 0,
          "target": 2
      },...
       ],
  "layouts": [
   ----------------------------------------
   contains all layouts of the project
   only contains nodes and links as well as colors specific to the layout
   ----------------------------------------
       {  "layoutname" : "name of first layout",
          "nodes": [
              {
                  "nodecolor": [
                      255,
                      35,
                      0,
                      120
                  ],
                  "pos": [
                      -0.5618057865250979,
                      0.1467411221839164,
                      0.49656801102094605
                  ],
                  "id": 0
               },...
        	],
          "links": [
              {
                  "linkcolor": [
                      0,
                      255,
                      0,
                      100
                  ],
                  "source": 0,
                  "target": 1
              },...
         	],
   	  }, {
          "layoutname" : "name of second layout",
          "nodes": [
              {
                  "nodecolor": "#0000ffaa",
                  "pos": [
                      -0.35948900932978317,
                      0.6255258442839948,
                      -0.04209289102217994
                  ],
                  "cluster": "cluster group 1",
                  "id": 0
               },... 
],
          "links": [
              {
                  "linkcolor": "#0000ff",
                  "source": 0,
                  "target": 1
              },
],
  	   }, { . . .  
 	},
}

'''

## CREATE A JSON FILE WITH THE ABOVE STRUCTURE to then create a project

In [28]:
import networkx as nx
import json 
import os

# these are the two functions one needs to create a JSON file to upload and then create the project in the backend 
import nx2json as nx2j 
import uploaderGraph as uG

In [14]:
# ----------------------------------------
# CREATE Json file
# ----------------------------------------
merged_graphs = nx2j.make_json(Graphs)
path = "temp_files/"

# save the merged graph in a json file
with open(path+Graphs[0].graph['projectname']+'.json', 'w') as fp:
    json.dump(merged_graphs, fp, indent=4)

In [ ]:
# ----------------------------------------
# READ Json file
# ----------------------------------------
filename = 'myfile.json'
currentwd = '.../DataDiVR_Webapp/temp_files/' # modify file location here
path = os.path.join(currentwd, filename)

# open the json file
with open(path, 'r') as f:
     G_merged = json.load(f)

In [ ]:
## ----------------------------------------
# CREATE A PROJECT for the VR Platform 
# ----------------------------------------
#the actual "upload step" to create a project with the required VR platform files 

uG.upload_filesJSON(G_merged)